# insurance-optimise

**Constrained portfolio pricing optimisation with FCA PS21/5 (ENBP) compliance.**

You have a technical model. You have elasticity estimates. The question is: what set of price multipliers maximises expected profit subject to FCA ENBP constraints, loss ratio bounds, retention floors, and rate change limits — simultaneously? This notebook answers that.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/burning-cost/insurance-optimise/blob/main/notebooks/quickstart.ipynb)

In [ ]:
!pip install -q insurance-optimise polars numpy

## Synthetic UK motor renewal book

500 renewal policies. In production, `technical_price` and `expected_loss_cost` come from your GLM/GBM, `elasticity` from `insurance-demand`'s `ElasticityEstimator`, and `enbp` from your rating engine's new-business quote for the same risk.

Key regulatory constraint: FCA PS21/5 (ENBP) — renewal premiums cannot exceed what a new customer would be quoted for the same risk and channel.

In [ ]:
import numpy as np
import polars as pl

rng = np.random.default_rng(42)
n = 500

technical_price    = rng.uniform(300, 1200, n)
expected_loss_cost = technical_price * rng.uniform(0.55, 0.75, n)
p_renewal          = rng.uniform(0.70, 0.95, n)         # renewal probability at current price
price_elasticity   = rng.uniform(-2.5, -0.8, n)         # from insurance-demand
is_renewal         = rng.choice([True, False], n, p=[0.7, 0.3])
enbp               = technical_price * rng.uniform(1.05, 1.25, n)  # ENBP ceiling per policy

print(f"Portfolio: {n} policies")
print(f"Technical price range: £{technical_price.min():.0f} – £{technical_price.max():.0f}")
print(f"Mean price elasticity: {price_elasticity.mean():.2f}")
print(f"Mean expected loss ratio: {(expected_loss_cost / technical_price).mean():.2%}")

## Run the optimiser

`PortfolioOptimiser` maximises expected profit (renewal probability × margin) using SLSQP with analytical gradients. Analytical gradients are essential — without them SLSQP runs 2N extra evaluations per iteration, which is prohibitively slow for a real book.

The `ConstraintConfig` encodes the hard constraints. ENBP is enforced at the per-policy level — the audit trail records whether it was binding for each renewal.

In [ ]:
from insurance_optimise import PortfolioOptimiser, ConstraintConfig

config = ConstraintConfig(
    lr_max=0.72,             # max loss ratio target
    retention_min=0.84,      # renewal retention floor
    max_rate_change=0.20,    # cap any single policy at +/-20%
    enbp_buffer=0.01,        # 1% safety margin below ENBP ceiling
    technical_floor=True,    # enforce price >= cost
)

opt = PortfolioOptimiser(
    technical_price=technical_price,
    expected_loss_cost=expected_loss_cost,
    p_demand=p_renewal,
    elasticity=price_elasticity,
    renewal_flag=is_renewal,
    enbp=enbp,
    constraints=config,
)

result = opt.optimise()
print(result)

## Inspect the optimal premiums

In [ ]:
df_result = pl.DataFrame({
    "technical_price":    technical_price.tolist(),
    "optimal_premium":    result.new_premiums.tolist(),
    "optimal_multiplier": result.multipliers.tolist(),
    "enbp_ceiling":       enbp.tolist(),
})

print("Summary statistics:")
print(df_result.describe())

# How many renewal policies hit the ENBP ceiling?
enbp_binding = (
    df_result
    .filter(pl.col("optimal_premium") >= pl.col("enbp_ceiling") * 0.99)
    .height
)
print(f"\nENBP binding for {enbp_binding} / {n} policies ({enbp_binding/n:.0%})")

## Scenario analysis

Elasticity estimates carry uncertainty. Run under pessimistic/central/optimistic scenarios and report the spread — this is the conversation the pricing committee needs to have.

In [ ]:
result_scenarios = opt.optimise_scenarios(
    elasticity_scenarios=[
        price_elasticity * 0.75,   # pessimistic: customers more price-sensitive
        price_elasticity,          # central estimate
        price_elasticity * 1.25,   # optimistic: customers less price-sensitive
    ],
    scenario_names=["pessimistic", "central", "optimistic"],
)
print(result_scenarios.summary())

## What you should see

- Optimiser result with expected profit, GWP, and loss ratio. If it reports NOT CONVERGED, the solution may still be useful but treat with caution — try reducing `n` or relaxing constraints.
- Optimal multipliers: typically 0.95–1.20 depending on where elasticity and ENBP constraints sit.
- Scenario summary: profit range across the three elasticity assumptions — this quantifies how sensitive the strategy is to estimation error.
- ENBP binding count: how many renewal policies are priced at the maximum permissible premium.

## Next steps

- **`EfficientFrontier`** — sweep retention vs profit to give the pricing team the full trade-off curve
- **`result.save_audit("audit.json")`** — FCA-ready JSON audit trail recording ENBP constraint status per policy
- **`stochastic_lr=True`** — Branda 2014 chance constraint on loss ratio under claims volatility

**GitHub:** https://github.com/burning-cost/insurance-optimise  
**PyPI:** https://pypi.org/project/insurance-optimise/